# Model comparison: Keras CNN vs JAX CNN vs ViT

**Hypothesis:** a ViT beats a CNN at detecting bars (`t03`) and estimating spiral-arm tightness
(`t10`).

This notebook trains nothing. It reads the test-set predictions and training histories that the
other notebooks save, and makes the figures for presenting the result:

| Model | Notebook | Role |
|---|---|---|
| Keras CNN | `02_cnn_baseline.ipynb` | The original baseline (Adam, one seed) |
| JAX CNN | `03_jax_cnn.ipynb` | **The controlled CNN:** same training recipe as the ViT |
| ViT | `04_vit.ipynb` | The challenger |

The hypothesis is tested on **ViT vs JAX CNN**, because those two differ only in the model class.
The Keras CNN is context: it shows the JAX CNN is a fair, competitive CNN rather than a weak one.

**Inputs** (all on the same test galaxies):

- `output/predictions/{keras_cnn,jax_cnn,vit}_seed*.csv`, written by each notebook. The Keras
  ones are made here from `models/cnn_baseline.keras` if they don't exist yet.
- `output/history/*_seed*.csv` for the learning curves (optional).

Figures are saved to `output/figures/` at presentation resolution.

## Colab setup

On Colab this clones the repo, downloads the data and installs what Colab lacks (a few minutes,
since Colab's disk is wiped between sessions). Locally it does nothing. Copy the `output/` and
`models/` folders from your training sessions into `/content/ai-570/` before running the rest.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !test -d /content/ai-570 || git clone -q https://github.com/joshbrgs/ai-570.git /content/ai-570
    %cd /content/ai-570
    !pip install -q dvc "flax>=0.12" optax
    !dvc update data/raw/*.dvc
    !python scripts/extract_data.py
    sys.path.append("/content/ai-570")  # makes `import gz2` work

In [ ]:
import os

# Run the saved Keras model on the JAX backend: gz2.pipeline hides the GPU from TensorFlow.
os.environ.setdefault("KERAS_BACKEND", "jax")

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve

from gz2.data import ROOT, WINDING_COLS
from gz2.evaluate import (
    PRED_COLS,
    TRUE_COLS,
    breakdown,
    load_context,
    load_histories,
    load_predictions,
    metrics,
    paired_bootstrap,
    save_predictions,
)
from gz2.pipeline import load_split, make_dataset

MODELS = {"keras_cnn": "Keras CNN", "jax_cnn": "JAX CNN", "vit": "ViT"}
# The two models the hypothesis is about get the strong, colourblind-safe blue/orange pair; the
# Keras reference is a recessive grey.
COLORS = {"keras_cnn": "#8f8d87", "jax_cnn": "#2a78d6", "vit": "#eb6834"}
PARAMS = {"jax_cnn": 422_308, "vit": 588_420}  # printed by 03 and 04; Keras is counted below
CNN, VIT = "jax_cnn", "vit"  # the controlled pair the hypothesis is tested on

KERAS_MODEL_PATH = ROOT / "models" / "cnn_baseline.keras"
KERAS_SEED = 42  # SEED in 02_cnn_baseline.ipynb
TARGETS = {"bar_acc": 0.90, "winding_rmse": 0.10}
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

INK, MUTED, GRID = "#2b2b2b", "#8f8d87", "#e6e5e0"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "axes.axisbelow": True, "axes.titlelocation": "left", "axes.titlesize": 12, "font.size": 10,
})


def save(fig, name: str) -> None:
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

## Load predictions

The Keras notebook only saves the trained model, so its test predictions are made here once,
through the same `gz2.pipeline` preprocessing the JAX models use, and cached in
`output/predictions/`.

In [ ]:
def predict_keras(model: keras.Model, ds) -> pd.DataFrame:
    """Same predictions format as gz2.jax_train.predict, from the two-head Keras model."""
    parts = []
    for batch in ds.as_numpy_iterator():
        out = model(batch["image"], training=False)
        part = pd.DataFrame({
            "dr7objid": batch["dr7objid"],
            "bar_true": batch["bar"],
            "bar_w": batch["bar_w"],
            "bar_prob": np.asarray(out["bar"])[:, 0],
            "winding_w": batch["winding_w"],
        })
        part[TRUE_COLS] = batch["winding"]
        part[PRED_COLS] = np.asarray(out["winding"])
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


split = load_split()
if KERAS_MODEL_PATH.exists():
    keras_model = keras.models.load_model(KERAS_MODEL_PATH)
    PARAMS["keras_cnn"] = keras_model.count_params()
    if not load_predictions("keras_cnn"):
        test_ds = make_dataset(split[split["split"] == "test"], training=False)
        print("saved ->", save_predictions(predict_keras(keras_model, test_ds), "keras_cnn", KERAS_SEED))

runs = {key: load_predictions(key) for key in MODELS}
for key in [k for k, r in runs.items() if not r]:
    print(f"No predictions for {MODELS[key]} yet: run its notebook first. Skipping it.")
    del runs[key]
pd.DataFrame({MODELS[k]: {"seeds": len(r), "test galaxies": len(r[0])} for k, r in runs.items()}).T

## Headline numbers

Mean ± standard deviation over seeds, next to the trivial baselines (predict the training set's
majority bar class and mean winding fractions) and the proposal targets. "Confident" keeps only
galaxies where volunteers agreed at ≥ 80%, which is how Zoobot reports its headline numbers.

In [ ]:
train = split[split["split"] == "train"]
majority = float(train.loc[train["bar_weight"] == 1, "bar_label"].mean() >= 0.5)
mean_winding = train.loc[train["winding_weight"] == 1, WINDING_COLS].mean().to_numpy()
reference = next(iter(runs.values()))[0]
bar_rows, winding_rows = reference[reference["bar_w"] > 0], reference[reference["winding_w"] > 0]
BASELINE = {
    "bar_acc": float((bar_rows["bar_true"] == majority).mean()),
    "winding_rmse": float(np.sqrt(((winding_rows[TRUE_COLS].to_numpy() - mean_winding) ** 2).mean())),
}

per_seed = pd.DataFrame([
    {"key": key, "seed": i, **metrics(preds)} for key, preds_list in runs.items() for i, preds in enumerate(preds_list)
])
context = load_context()
breakdowns = {
    key: pd.concat([breakdown(p, context) for p in preds_list]).groupby(level=0, sort=False).mean()
    for key, preds_list in runs.items()
}
confident_row = next(r for r in next(iter(breakdowns.values())).index if r.startswith("confident"))


def mean_std(key: str, metric: str) -> str:
    values = per_seed.loc[per_seed["key"] == key, metric]
    return f"{values.mean():.3f}" + (f" ± {values.std():.3f}" if len(values) > 1 else "")


headline = pd.DataFrame({
    MODELS[key]: {
        "parameters": f"{PARAMS[key]:,}" if key in PARAMS else "–",
        "seeds": str(len(runs[key])),
        "bar accuracy": mean_std(key, "bar_acc"),
        "bar AUC": mean_std(key, "bar_auc"),
        "bar accuracy (confident)": f"{breakdowns[key].loc[confident_row, 'bar_acc']:.3f}",
        "winding RMSE": mean_std(key, "winding_rmse"),
        "winding RMSE (confident)": f"{breakdowns[key].loc[confident_row, 'winding_rmse']:.3f}",
    }
    for key in runs
})
headline["trivial baseline"] = {"bar accuracy": f"{BASELINE['bar_acc']:.3f}", "winding RMSE": f"{BASELINE['winding_rmse']:.3f}"}
headline["target"] = {"bar accuracy": "0.900", "winding RMSE": "0.100"}
headline.fillna("")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
panels = [("bar_acc", "Bar accuracy (higher is better)"), ("winding_rmse", "Winding RMSE (lower is better)")]
for ax, (metric, title) in zip(axes, panels):
    stats = per_seed.groupby("key", sort=False)[metric].agg(["mean", "std"]).fillna(0.0)
    for y, (key, row) in enumerate(stats.iterrows()):
        ax.errorbar(row["mean"], y, xerr=row["std"], fmt="o", color=COLORS[key], markersize=9,
                    capsize=4, linewidth=2)
        ax.annotate(f"{row['mean']:.3f}", (row["mean"], y), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=9, color=INK)
    for x, label, style in [(BASELINE[metric], "trivial baseline", "--"), (TARGETS[metric], "target", ":")]:
        ax.axvline(x, color=MUTED if style == "--" else INK, linestyle=style, linewidth=1.5)
        ax.text(x, 1.01, label, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=8,
                color=MUTED)
    ax.set_yticks(range(len(stats)), [MODELS[k] for k in stats.index])
    ax.set_ylim(-0.6, len(stats) - 0.4)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_title(title, pad=16)
fig.tight_layout()
save(fig, "compare_headline")
plt.show()

## Is the gap real? Paired bootstrap

Each galaxy's error is averaged over seeds, then the test galaxies are resampled 1,000 times,
jointly for both models. A difference whose 95% interval excludes 0 is unlikely to be test-set
noise; those points are drawn solid, and the rest hollow.

The **first row is the hypothesis test** (ViT − JAX CNN). For bar accuracy, a positive difference
favours the second model named; for winding RMSE, a negative one does.

In [ ]:
PAIRS = [(a, b) for a, b in [(CNN, VIT), ("keras_cnn", VIT), ("keras_cnn", CNN)] if a in runs and b in runs]
bootstrap = pd.concat({f"{MODELS[b]} − {MODELS[a]}": paired_bootstrap(runs[a], runs[b]) for a, b in PAIRS}) if PAIRS else None

if bootstrap is None:
    print("The paired comparison needs predictions from at least two models.")
else:
    bootstrap["significant"] = (bootstrap["95% CI low"] > 0) | (bootstrap["95% CI high"] < 0)
    fig, axes = plt.subplots(1, 2, figsize=(11, 0.8 + 0.7 * len(PAIRS)))
    panels = [("bar accuracy", "Bar accuracy difference (> 0: first model better)"),
              ("winding RMSE", "Winding RMSE difference (< 0: first model better)")]
    for ax, (metric, title) in zip(axes, panels):
        rows = bootstrap.xs(metric, level=1)
        for y, (pair, row) in enumerate(rows.iterrows()):
            color = INK if pair.startswith(MODELS[VIT]) else MUTED
            ax.errorbar(row["B − A"], y, xerr=[[row["B − A"] - row["95% CI low"]], [row["95% CI high"] - row["B − A"]]],
                        fmt="o", color=color, markersize=8, capsize=4, linewidth=2,
                        markerfacecolor=color if row["significant"] else "white")
        ax.axvline(0, color=MUTED, linewidth=1)
        ax.set_yticks(range(len(rows)), rows.index)
        ax.set_ylim(-0.6, len(rows) - 0.4)
        ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        ax.set_title(title)
    fig.tight_layout()
    save(fig, "compare_bootstrap")
    plt.show()

bootstrap.style.format({c: "{:.4f}" for c in ["A", "B", "B − A", "95% CI low", "95% CI high"]} | {"n galaxies": "{:,.0f}"}) if bootstrap is not None else None

## Where does each model win? Performance by galaxy size

Test galaxies binned into quintiles of their size in original SDSS pixels. The EDA showed labels
skew toward large, well-resolved galaxies; if the ViT's advantage (or disadvantage) only appears
in one end of this range, that's a finding in itself. Values are averaged over seeds.

In [ ]:
size_rows = [r for r in next(iter(breakdowns.values())).index if r.startswith("size")]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
panels = [("bar_acc", "Bar accuracy by galaxy size"), ("winding_rmse", "Winding RMSE by galaxy size")]
for ax, (metric, title) in zip(axes, panels):
    for key, table in breakdowns.items():
        ax.plot(range(len(size_rows)), table.loc[size_rows, metric], color=COLORS[key], linewidth=2,
                marker="o", markersize=8, label=MODELS[key])
    ax.set_xticks(range(len(size_rows)), [r.removeprefix("size ") for r in size_rows])
    ax.set_xlabel("Galaxy diameter in original SDSS pixels (quintiles)")
    ax.set_title(title)
axes[0].legend(frameon=False)
fig.tight_layout()
save(fig, "compare_by_size")
plt.show()

pd.concat({MODELS[k]: t[["n_bar", "bar_acc", "n_winding", "winding_rmse"]] for k, t in breakdowns.items()}, axis=1).style.format(
    lambda v: f"{v:,.0f}" if v > 1 else f"{v:.3f}"
)

## Learning curves

Validation bar accuracy and winding RMSE per epoch: the mean over seeds as a thick line, each seed
as a thin one. Total loss isn't shown because Keras and the JAX loop normalise masked losses
differently, so their loss values aren't on the same scale; these two metrics are.

In [ ]:
histories = {key: pd.concat(h) for key in MODELS if (h := load_histories(key))}
if not histories:
    print("No training histories saved yet: re-run the training notebooks to get learning curves.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    panels = [("val_bar_acc", "Validation bar accuracy"), ("val_winding_rmse", "Validation winding RMSE")]
    for ax, (column, title) in zip(axes, panels):
        for key, hist in histories.items():
            for _, seed_hist in hist.groupby("seed"):
                ax.plot(seed_hist["epoch"], seed_hist[column], color=COLORS[key], linewidth=1, alpha=0.35)
            mean = hist.groupby("epoch")[column].mean()
            ax.plot(mean.index, mean, color=COLORS[key], linewidth=2.5, label=MODELS[key])
        ax.set_xlabel("Epoch")
        ax.set_title(title)
    axes[0].legend(frameon=False)
    fig.tight_layout()
    save(fig, "compare_learning_curves")
    plt.show()

## Bar detection: ROC curves

Accuracy depends on the 0.5 threshold; the ROC curve shows how well each model ranks barred above
unbarred galaxies at every threshold. Probabilities are averaged over seeds.

In [ ]:
mean_bar = {
    key: pd.concat(preds_list).groupby("dr7objid")[["bar_prob", "bar_true", "bar_w"]].mean().query("bar_w > 0")
    for key, preds_list in runs.items()
}
fig, ax = plt.subplots(figsize=(5, 5))
for key, df in mean_bar.items():
    fpr, tpr, _ = roc_curve(df["bar_true"], df["bar_prob"])
    auc = roc_auc_score(df["bar_true"], df["bar_prob"])
    ax.plot(fpr, tpr, color=COLORS[key], linewidth=2, label=f"{MODELS[key]} (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Bar detection ROC")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save(fig, "compare_bar_roc")
plt.show()

## Galaxies where the ViT and the CNN disagree

Bar predictions where exactly one of the controlled pair is right, the five most decisive each
way. Images are shown as the models saw them (centre crop). Good for a slide: it shows *what*
each architecture gets right, not just how often.

In [ ]:
if CNN in mean_bar and VIT in mean_bar:
    pair = mean_bar[CNN].join(mean_bar[VIT][["bar_prob"]], rsuffix="_vit").rename(columns={"bar_prob": "bar_prob_cnn"})
    truth = pair["bar_true"] == 1
    cnn_right = (pair["bar_prob_cnn"] >= 0.5) == truth
    vit_right = (pair["bar_prob_vit"] >= 0.5) == truth
    pair["gap"] = (pair["bar_prob_vit"] - pair["bar_prob_cnn"]).abs()
    groups = {
        f"ViT right, CNN wrong ({(vit_right & ~cnn_right).sum():,} galaxies)": pair[vit_right & ~cnn_right],
        f"CNN right, ViT wrong ({(cnn_right & ~vit_right).sum():,} galaxies)": pair[cnn_right & ~vit_right],
    }
    image_paths = split.set_index("dr7objid")["image_path"]

    fig = plt.figure(figsize=(12, 5.6), layout="constrained")
    for subfig, (title, rows) in zip(fig.subfigures(2, 1), groups.items()):
        subfig.suptitle(title, x=0.01, ha="left", fontsize=12)
        for ax, (objid, row) in zip(subfig.subplots(1, 5), rows.nlargest(5, "gap").iterrows()):
            image = Image.open(image_paths[objid])
            w, h = image.size
            ax.imshow(image.crop((w // 4, h // 4, 3 * w // 4, 3 * h // 4)))
            ax.set_title(
                f"{'barred' if row['bar_true'] else 'no bar'}\nViT {row['bar_prob_vit']:.2f} · CNN {row['bar_prob_cnn']:.2f}",
                fontsize=9,
            )
            ax.axis("off")
    save(fig, "compare_disagreements")
    plt.show()

## Reading the result

**The ViT beats the CNN on a task** if, for that task, the ViT − JAX CNN interval in the bootstrap
figure excludes 0 in the ViT's favour *and* the gap is bigger than the seed-to-seed spread in the
headline figure. Check bars and winding separately: the answer can differ between them.

**Context from the literature** (not directly comparable: different survey, labels or models):

| Source | Bar accuracy | Winding accuracy |
|---|---|---|
| Zoobot (Walmsley et al. 2022, DECaLS), all galaxies | 81.9% | 78.3% |
| Zoobot, confident galaxies | 99.5% | 99.1% |

Zoobot reports winding as accuracy of the dominant answer; ours is RMSE on the vote fractions,
with a target of 0.10. Its "confident" rows correspond to our confident rows above.

**Findings** *(fill in after training)*:

- Bars: …
- Winding: …
- By galaxy size: …